<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

-Juan Pablo Sanchez Luis

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.4f}'.format

In [3]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [5]:
df = pd.read_csv('/content/alimentos_retail_top30.csv', encoding='latin1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÃZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÃZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÃ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia


In [6]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [7]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [8]:
print(df.isnull().sum())

InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     20
Country         0
dtype: int64


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [9]:
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

En la columna CustomerID se identifica que hay valores nulos (el count es 6,879 mientras que las demás columnas tienen 6,899 filas), lo que significa que 20 transacciones no tienen cliente identificado y deben eliminarse ya que no podemos agrupar compras sin saber a quién pertenecen.
En la columna Quantity se identifica que el valor mínimo es -5, lo que indica la presencia de devoluciones o cancelaciones. Estas cantidades negativas no representan compras reales y deben filtrarse antes del análisis, ya que distorsionarían los patrones de consumo que buscamos descubrir con el algoritmo Apriori.

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [10]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
df_limpio = df.dropna(subset=['CustomerID'])
print(f"Filas antes: {len(df)} | Filas después: {len(df_limpio)}")

Filas antes: 6899 | Filas después: 6879


In [11]:
# El tipo de dato de CustomerID debe ser entero
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)


2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [12]:
# TAREA: # Verifica cuántas descripciones únicas hay.
print(f"Descripciones únicas antes de limpieza: {df['Description'].str.strip().nunique()}")


Descripciones únicas antes de limpieza: 20


In [13]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [14]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
print(f"Descripciones únicas: {df_limpio['Description'].nunique()}")

Descripciones únicas: 20


2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [15]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio = df_limpio[~df_limpio['InvoiceNo'].astype(str).str.startswith('C')]

In [16]:
# TAREA: Elimina las filas con cantidades negativas.
df_limpio = df_limpio[df_limpio['Quantity'] > 0]


In [17]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [18]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [19]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
cesta_mx = preparar_cesta(df_limpio, 'Mexico')

In [23]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
cesta_mx = preparar_cesta(df_limpio, 'MÃ©xico')

frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx = frequent_itemsets_mx.sort_values('support', ascending=False)
frequent_itemsets_mx.head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÃO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÃZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÃN)
9,0.33,(TOTOPOS)
31,0.33,"(CHILE JALAPEÃO, TOMATE)"


In [24]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
rules_mx = association_rules(frequent_itemsets_mx, metric='lift', min_threshold=2)

In [25]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_mx = rules_mx.sort_values(['lift', 'confidence'], ascending=False).reset_index(drop=True)
rules_mx[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']].head(10)


,antecedents,consequents,antecedent support,consequent support,confidence,lift
0,"(CHILE JALAPEÃO, CEBOLLA)","(TOMATE, CILANTRO)",0.32,0.32,0.92,2.90
1,"(TOMATE, CILANTRO)","(CHILE JALAPEÃO, CEBOLLA)",0.32,0.32,0.93,2.90
2,"(CEBOLLA, CILANTRO)","(CHILE JALAPEÃO, TOMATE)",0.31,0.33,0.95,2.88
3,"(CHILE JALAPEÃO, TOMATE)","(CEBOLLA, CILANTRO)",0.33,0.31,0.90,2.88
4,"(AGUACATE, LIMÃN)",(TOTOPOS),0.27,0.33,0.93,2.82
5,(TOTOPOS),"(AGUACATE, LIMÃN)",0.33,0.27,0.75,2.82
6,"(CHILE JALAPEÃO, CILANTRO)","(CEBOLLA, TOMATE)",0.32,0.33,0.92,2.81
7,"(CEBOLLA, TOMATE)","(CHILE JALAPEÃO, CILANTRO)",0.33,0.32,0.91,2.81
8,"(TOMATE, AGUACATE, LIMÃN)",(TOTOPOS),0.02,0.33,0.91,2.76
9,(TOTOPOS),"(TOMATE, AGUACATE, LIMÃN)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Regla 1: {Chile Jalapeño, Cebolla} → {Tomate, Cilantro}
Esta regla revela un patrón clásico de la cocina mexicana: los clientes que compran chile jalapeño y cebolla casi siempre también compran tomate y cilantro. Son los 4 ingredientes base de la salsa mexicana tradicional, lo que indica que los clientes compran en conjunto los ingredientes para preparar salsas y guisos.
Regla 3: {Chile Jalapeño, Tomate} → {Cebolla, Cilantro}
Mismo patrón desde otro ángulo: quien compra chile jalapeño y tomate complementa su compra con cebolla y cilantro. Confirma que estos 4 productos forman un bloque de compra prácticamente inseparable en el mercado mexicano.
Regla 5: {Totopos} → {Aguacate, Limón}
Los clientes que compran totopos casi siempre añaden aguacate y limón al carrito, la combinación perfecta para preparar guacamole. Son productos de snack y botana que se consumen juntos de forma natural.


3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1: {Chile Jalapeño, Cebolla} → {Tomate, Cilantro}

Soporte antecedente (0.32): el 32% de las transacciones en México incluyen chile jalapeño y cebolla juntos.
Soporte consecuente (0.32): el 32% de las transacciones incluyen tomate y cilantro juntos.
Confianza (0.92): cuando un cliente compra chile jalapeño y cebolla, en el 92% de los casos también compra tomate y cilantro. Es una asociación muy fuerte.
Lift (2.90): la probabilidad de comprar tomate y cilantro es 2.9 veces mayor cuando el cliente ya lleva chile jalapeño y cebolla, comparado con comprarlos al azar.

Regla 3: {Chile Jalapeño, Tomate} → {Cebolla, Cilantro}

Soporte antecedente (0.33): el 33% de las transacciones incluyen chile jalapeño y tomate.
Soporte consecuente (0.31): el 31% incluyen cebolla y cilantro.
Confianza (0.90): el 90% de quienes compran chile jalapeño y tomate también llevan cebolla y cilantro.
Lift (2.88): comprar cebolla y cilantro es 2.88 veces más probable cuando ya se llevan chile jalapeño y tomate.

Regla 5: {Totopos} → {Aguacate, Limón}

Soporte antecedente (0.33): el 33% de las transacciones en México incluyen totopos.
Soporte consecuente (0.27): el 27% incluyen aguacate y limón juntos.
Confianza (0.75): el 75% de quienes compran totopos también llevan aguacate y limón.
Lift (2.82): la probabilidad de comprar aguacate y limón es 2.82 veces mayor cuando el cliente ya lleva totopos.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Basado en estas reglas existen dos estrategias concretas:
Estrategia 1 - Bundle de Salsa Mexicana: Crear un paquete promocional que incluya Chile Jalapeño, Tomate, Cebolla y Cilantro con un descuento del 10-15%. Dado que el 90-92% de los clientes que compran dos de estos productos terminan llevando los cuatro, ofrecer el combo completo a precio especial aumentaría el ticket promedio y reduciría la fricción de búsqueda. Este bundle podría llamarse "Kit Salsa Casera".

Estrategia 2 - Cross-selling Guacamole: En la página de producto de los Totopos, mostrar aguacate y limón como "frecuentemente comprados juntos". Dado que el 75% de quienes compran totopos llevan estos productos, una sugerencia automática en el checkout capturaría ventas adicionales que de otro modo quedarían en el olvido. Este bundle podría llamarse "Kit Guacamole".

3.6 Análisis para Colombia

In [26]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.

cesta_co = preparar_cesta(df_limpio, 'Colombia')

In [27]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.

frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co = frequent_itemsets_co.sort_values('support', ascending=False)
frequent_itemsets_co.head(10)


,support,itemsets
3,0.41,(CAFÃ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÃCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÃZ)
37,0.32,"(CAFÃ, LECHE)"
31,0.32,"(AZÃCAR, LECHE)"


In [28]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
rules_co = association_rules(frequent_itemsets_co, metric='lift', min_threshold=2)

In [29]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_co = rules_co.sort_values(['lift', 'confidence'], ascending=False).reset_index(drop=True)
rules_co[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']].head(10)


,antecedents,consequents,antecedent support,consequent support,confidence,lift
0,"(AZÃCAR, FRIJOL CARGAMANTO, CAFÃ)",(LECHE),0.05,0.39,0.96,2.44
1,(LECHE),"(AZÃCAR, FRIJOL CARGAMANTO, CAFÃ)",0.39,0.05,0.11,2.44
2,"(AZÃCAR, CAFÃ)",(LECHE),0.32,0.39,0.95,2.41
3,(LECHE),"(AZÃCAR, CAFÃ)",0.39,0.32,0.76,2.41
4,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.91,2.41
5,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
6,"(PAN TAJADO, AZÃCAR, CAFÃ)",(LECHE),0.03,0.39,0.94,2.39
7,(LECHE),"(PAN TAJADO, AZÃCAR, CAFÃ)",0.39,0.03,0.07,2.39
8,"(HUEVOS, AZÃCAR, CAFÃ)",(LECHE),0.04,0.39,0.92,2.36
9,(LECHE),"(HUEVOS, AZÃCAR, CAFÃ)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?
Regla 1: {Azúcar, Frijol Cargamanto, Café} → {Leche}
Esta regla revela un patrón típico del hogar colombiano: quien compra azúcar, frijol y café casi siempre también compra leche. Son productos de la canasta básica familiar colombiana, donde el café con leche es una bebida cotidiana y el frijol cargamanto es un ingrediente central de la cocina paisa. Es completamente diferente a México, donde los patrones giraban en torno a ingredientes para salsas y botanas.

Regla 3: {Azúcar, Café} → {Leche}
Confirma el patrón anterior de forma más directa: azúcar y café van siempre acompañados de leche. El tinto con leche o el café con leche es una costumbre profundamente arraigada en la cultura colombiana, lo que hace que estos tres productos sean prácticamente inseparables en el carrito de compras.

Regla 5: {Arroz} → {Frijol Cargamanto, Aceite de Girasol}
Quien compra arroz casi siempre lleva también frijol cargamanto y aceite de girasol. Esta es la combinación clásica del almuerzo colombiano: arroz con frijoles salteados en aceite, el plato más común de la gastronomía popular colombiana. Confirma que los patrones de compra están fuertemente ligados a los hábitos alimenticios culturales del país.

Diferencia con México: Mientras México muestra patrones de compra alrededor de ingredientes frescos para salsas y botanas (tomate, cilantro, jalapeño, aguacate), Colombia muestra patrones de canasta básica de despensa (leche, café, azúcar, arroz, frijol), lo que sugiere que los clientes colombianos hacen compras más orientadas al hogar y la alimentación diaria.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1: {Azúcar, Frijol Cargamanto, Café} → {Leche}

Soporte antecedente (0.05): el 5% de las transacciones en Colombia incluyen azúcar, frijol y café juntos.
Soporte consecuente (0.39): el 39% de las transacciones incluyen leche, lo que la convierte en el producto más comprado del mercado colombiano.
Confianza (0.96): cuando un cliente compra azúcar, frijol y café, en el 96% de los casos también lleva leche. Es la confianza más alta de todas las reglas.
Lift (2.44): la probabilidad de comprar leche es 2.44 veces mayor cuando el cliente ya lleva azúcar, frijol y café.

Regla 3: {Azúcar, Café} → {Leche}

Soporte antecedente (0.32): el 32% de las transacciones incluyen azúcar y café juntos, una combinación muy frecuente.
Soporte consecuente (0.39): el 39% de las transacciones incluyen leche.
Confianza (0.95): el 95% de quienes compran azúcar y café también llevan leche.
Lift (2.41): comprar leche es 2.41 veces más probable cuando el cliente ya lleva azúcar y café.

Regla 5: {Arroz} → {Frijol Cargamanto, Aceite de Girasol}

Soporte antecedente (0.38): el 38% de las transacciones en Colombia incluyen arroz, siendo uno de los productos más comprados.
Soporte consecuente (0.30): el 30% incluyen frijol cargamanto y aceite de girasol juntos.
Confianza (0.73): el 73% de quienes compran arroz también llevan frijol y aceite de girasol.
Lift (2.41): la probabilidad de comprar frijol y aceite es 2.41 veces mayor cuando el cliente ya lleva arroz.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

Estrategia 1 - Bundle Desayuno Colombiano: Crear un paquete promocional que incluya Café, Leche, Azúcar y Pan Tajado con un descuento especial. Dado que el 95-96% de quienes compran café y azúcar también llevan leche, y la regla 6 muestra que el pan tajado también entra en este patrón, un "Kit Desayuno Colombiano" con precio especial aumentaría el ticket promedio aprovechando un hábito de consumo diario ya establecido.

Estrategia 2 - Bundle Almuerzo Tradicional: Ofrecer una promoción conjunta de Arroz, Frijol Cargamanto y Aceite de Girasol bajo el nombre "Combo Bandeja" o "Kit Almuerzo". Con una confianza del 73% entre arroz y los otros dos productos, mostrar estos artículos como sugeridos en el checkout o agruparlos físicamente en la plataforma aumentaría las ventas cruzadas de forma natural, aprovechando el hábito del almuerzo colombiano tradicional.